# Phase 1: Ingestion
Parse raw scorecard JSON into structured ParsedContext.

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [4]:
from scripts.ingestion.ingestor import ingest_from_file, save_context

input_path = ROOT / 'data' / 'input' / 'aggregated_scorecard_output.json'
NB_DATA = ROOT / 'notebooks' / 'data'
NB_DATA.mkdir(exist_ok=True)

ctx = ingest_from_file(input_path)
save_context(ctx, NB_DATA / 'parsed_context.json')

print(f'Agent:      {ctx.meta["agent_name"]}')
print(f'Agent ID:   {ctx.meta["agent_id"]}')
print(f'Date:       {ctx.meta["certification_date"]}')
print(f'Total runs: {ctx.meta["total_runs"]}')
print(f'Categories: {len(ctx.categories)}')
print(f'Warnings:   {len(ctx.warnings)}')

Agent:      Flash Agent
Agent ID:   flash-001-abc123
Date:       2026-03-08
Total runs: 15
Categories: 3
Warnings:   2


In [5]:
import json

raw = json.loads(input_path.read_text(encoding='utf-8'))
parsed = json.loads((NB_DATA / 'parsed_context.json').read_text(encoding='utf-8'))

print('=== Validation: Parsed vs Input ===')
checks = [
    ('agent_name',         parsed['meta']['agent_name'],         raw['agent_name']),
    ('agent_id',           parsed['meta']['agent_id'],           raw['agent_id']),
    ('total_runs',         parsed['meta']['total_runs'],         raw['total_runs']),
    ('total_faults',       parsed['meta']['total_faults_tested'],raw['total_faults_tested']),
    ('total_categories',   parsed['meta']['total_fault_categories'], raw['total_fault_categories']),
    ('category_count',     len(parsed['categories']),            len(raw['fault_category_scorecards'])),
]

all_ok = True
for label, got, expected in checks:
    ok = 'PASS' if got == expected else 'FAIL'
    if ok == 'FAIL':
        all_ok = False
    print(f'  {ok} {label}: {got} (expected {expected})')

# Validate per-category runs
for i, cat in enumerate(parsed['categories']):
    raw_cat = raw['fault_category_scorecards'][i]
    runs_ok = cat['total_runs'] == raw_cat['total_runs']
    ok = 'PASS' if runs_ok else 'FAIL'
    if not runs_ok:
        all_ok = False
    print(f'  {ok} {cat["label"]} runs: {cat["total_runs"]} (expected {raw_cat["total_runs"]})')

print(f'\nResult: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"}')

=== Validation: Parsed vs Input ===
  PASS agent_name: Flash Agent (expected Flash Agent)
  PASS agent_id: flash-001-abc123 (expected flash-001-abc123)
  PASS total_runs: 15 (expected 15)
  PASS total_faults: 3 (expected 3)
  PASS total_categories: 3 (expected 3)
  PASS category_count: 3 (expected 3)
  PASS Application runs: 5 (expected 5)
  PASS Network runs: 5 (expected 5)
  PASS Resource runs: 5 (expected 5)

Result: ALL CHECKS PASSED


---
##  `Statistical_hypothesis` propagation

Confirm the new block flows from the aggregated scorecard into `ParsedContext` and that the read-only view helpers expose it correctly.

To exercise both branches, point `input_path` (cell 2) at:
- `cert_builder/data_certifier/scenario_1_out/aggregated_scorecard_output_sre-agent-v2.1.json` → `status="ok"` with full H01–H09 results
- `cert_builder/data_certifier/scenario_2_out/aggregated_scorecard_output_sre-agent-v2.1.json` → `status="skipped"`, `reason="insufficient_runs"`

In [8]:
from scripts.ingestion.ingestor import ingest_from_file, save_context

input_path = ROOT / 'data_certifier' / 'scenario_2_out' / 'aggregated_scorecard_output_sre-agent-v2.1.json'
NB_DATA = ROOT / 'notebooks' / 'data'
NB_DATA.mkdir(exist_ok=True)

ctx = ingest_from_file(input_path)
save_context(ctx, NB_DATA / 'parsed_context.json')

print(f'Agent:      {ctx.meta["agent_name"]}')
print(f'Agent ID:   {ctx.meta["agent_id"]}')
print(f'Date:       {ctx.meta["certification_date"]}')
print(f'Total runs: {ctx.meta["total_runs"]}')
print(f'Categories: {len(ctx.categories)}')
print(f'Warnings:   {len(ctx.warnings)}')

Agent:      SRE-Agent
Agent ID:   sre-agent-v2.1
Date:       2026-04-28
Total runs: 170
Categories: 3
Warnings:   0


In [9]:
# Re-ingest using the updated dataclass and inspect the new field.
ctx = ingest_from_file(input_path)

sh = ctx.statistical_hypothesis
print('statistical_hypothesis keys :', list(sh.keys()))
print('status                      :', sh.get('status'))
print('reason                      :', sh.get('reason'))
print('min_required                :', sh.get('min_required'))
print('observed_per_category       :', sh.get('observed_per_category'))
print('message                     :', sh.get('message'))
print('has results                 :', 'results' in sh)

statistical_hypothesis keys : ['status', 'reason', 'min_required', 'observed_per_category', 'message']
status                      : skipped
reason                      : insufficient_runs
min_required                : 30
observed_per_category       : {'application_fault': 10, 'network_fault': 80, 'resource_fault': 80}
message                     : Statistical hypothesis testing requires ≥30 runs per fault category. Section omitted; see Experiment Scope for details.
has results                 : False


In [10]:
# Exercise the hypothesis_view helper.
from scripts.ingestion import hypothesis_view as hv

print('status                :', hv.status(ctx))
print('is_ok                 :', hv.is_ok(ctx))
print('is_skipped            :', hv.is_skipped(ctx))
print('is_not_requested      :', hv.is_not_requested(ctx))
print('skip_reason           :', hv.skip_reason(ctx))
print('skip_message          :', hv.skip_message(ctx))
print('min_required          :', hv.min_required(ctx))
print('observed_per_category :', hv.observed_per_category(ctx))
res = hv.results(ctx)
print('results keys          :', list(res.keys()) if res else None)

status                : skipped
is_ok                 : False
is_skipped            : True
is_not_requested      : False
skip_reason           : insufficient_runs
skip_message          : Statistical hypothesis testing requires ≥30 runs per fault category. Section omitted; see Experiment Scope for details.
min_required          : 30
observed_per_category : {'application_fault': 10, 'network_fault': 80, 'resource_fault': 80}
results keys          : None


In [11]:
# Backwards-compat: a scorecard without the block must default to not_requested.
import json as _json
from scripts.ingestion.ingestor import ingest

raw_no_sh = _json.loads(input_path.read_text(encoding='utf-8'))
raw_no_sh.pop('statistical_hypothesis', None)
ctx_default = ingest(raw_no_sh)

assert hv.status(ctx_default) == 'not_requested'
assert hv.is_ok(ctx_default) is False
assert hv.is_skipped(ctx_default) is False
assert hv.results(ctx_default) is None
print('PASS  default block ->', ctx_default.statistical_hypothesis)

PASS  default block -> {'status': 'not_requested'}
